In [2]:
from gerrychain import Graph, Partition, MarkovChain, Election, updaters, constraints, accept
from gerrychain.proposals import recom
from gerrychain.constraints import contiguous
from gerrychain.tree import bipartition_tree
from functools import partial
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
# Load the graph in from the provided json file
graph = Graph.from_json("./PA_VTDs_clean.json")

# Set up election updater
election = Election("PRES12", {"Dem" : "PRES12D", "Rep": "PRES12R"})

# Set up updaters
my_updater = {
    "population": updaters.Tally("TOT_POP", alias="population"),
    "white_population": updaters.Tally("WHITE_POP", alias="white_population"),
    "black_population": updaters.Tally("BLACK_POP", alias="black_population"),
    "hispanic_population": updaters.Tally("HISP_POP", alias="hispanic_population"),
    "asian_population": updaters.Tally("ASIAN_POP", alias="asian_population"),
    "native_population": updaters.Tally("NATIVE_POP", alias="native_population"),
    "cut_edges": updaters.cut_edges,
    "PRES12" : election
}

# Set up the initial partition object
initial_partition = Partition(
    graph,
    assignment = "2011_PLA_1",
    updaters = my_updater
)

ideal_population = sum(initial_partition["population"].values()) / len(initial_partition)
# print(f"ideal population: {ideal_population}")
print(list(initial_partition.updaters.keys()))
# create new function from the recom with our given arguments
proposal = partial(
    recom,
    pop_col="TOT_POP",
    pop_target=ideal_population,
    epsilon=0.01,
    node_repeats=2,
    method = partial(
        bipartition_tree,
        max_attempts=100,
        allow_pair_reselection=True 
    )
)

['cut_edges', 'population', 'white_population', 'black_population', 'hispanic_population', 'asian_population', 'native_population', 'PRES12']


In [12]:
def vra_constrained(partition):
    return True

In [13]:
ensemble = []

BURN_IN = 1000           # discard first 1000 steps for randomization
THINNING = 25           # every 10th plan
TARGET_PLANS = 250      # plans in final ensemble

total_steps = BURN_IN + (THINNING * TARGET_PLANS)

chain = MarkovChain(
    proposal=proposal,
    constraints=[contiguous, vra_constrained],
    accept=accept.always_accept,
    initial_state=initial_partition,
    total_steps=total_steps
)